## Pegasus Evaluation: Depth

Section B.3. Setup, methods and metrics: `2a_base_case.ipynb`; parameters: `pegasus/evaluation_setup.py`.
B.3 runs on `tree_source`.

In [ ]:
# Data source of B.3:
# - "real": the fork-join trees measured on the second testbed machine (statics/results_milos_v2/b3-vm/, VM_TREE_SETS in pegasus/real_data.py)
# - "synthetic": balanced trees of the three generated services of B.1, with a simulated ground truth (pegasus/trees.py)
tree_source = "real"

# the parameters and helpers that every evaluation notebook shares, defined once in pegasus/evaluation_setup.py
from pegasus.evaluation_setup import *
import pandas as pd
from pegasus.figures import math_label, sweep_table, time_table
p_label = math_label(p_label)   # the quantile name in the math notation of the paper, e.g. $P_{95}$, for the figures and the text
# synthetic trees reuse the configurations and the training data of the synthetic base case of B.1, read from
# statics/evaluation_cache/ or computed here (b1_base); the measured trees need no base case
if tree_source == "synthetic":
    globals().update(b1_base("synthetic"))
print(f"{tree_source} trees, {len(seeds)} seeds")

### B.3 Varying Depth of a Map-Reduce Application

Fork-join trees: the leaves alternate between two detectors, every inner service is an aggregator; every frame goes to
all leaves, and the root releases it once both children of every aggregator have. A *balanced* tree with $h$ levels
(`tree_levels`) has $2^h - 1$ services, because every aggregator has two children, so level $i$ holds $2^{i-1}$ services.
Every method bounds each leaf-to-root path and takes the maximum over the paths (`compare_tree_methods`).

- **Real (`tree_source = "real"`):** the balanced trees measured on the second testbed machine, a virtual machine
  with 75 cores (`statics/results_milos_v2/b3-vm/`; object and face detector as leaves, image transformer as
  aggregator, one worker and 1.5 cores each, 8 configurations repeated 6 times per tree). The first machine had too
  few cores for the deeper trees: its services slowed each other down there, so their processing times were far
  from those of the benchmarks. Two tree sets (`VM_TREE_SETS` in `pegasus/real_data.py`), each with its own figure:
  - *Large trees* (the main set), image resolutions 288 to 640 px, $h$ = 1 to 6: 8 configurations drawn anew for every
    $h$. The arrival rates are low enough that the camera, which sends every frame to all leaves, keeps up at every $h$.
    A tree joins once the campaign has finished it (`vm_tree_experiments`).
  - *Small trees* (a check), 160 to 256 px, $h$ = 1 to 4: the same 8 configurations at every $h$. Deeper small trees
    are left out: at their higher arrival rates the camera falls behind in sending the frames to the 16 or 32 leaves.

  Every service type is trained on the grid benchmark of the same machine (`VM_TREE_BENCHMARKS`), which measured every
  configuration of the service alone: the face detector and the image transformer on the 1.5-core grid (`grid15-*`),
  whose processing times match those in the trees, and the object detector on the 8-core grid (`grid-*`), because the
  1.5-core grid does not contain it. The handling overhead of a service that receives its frames from another service
  is measured on the other trees of the set (`tree_handling_overhead`). Held-out runs: throttled for less than
  `TREE_THROTTLE_LIMIT` of their CPU periods at every node.

  Even on this machine, the services of the deepest trees process an item more slowly than in their benchmark
  (no method corrects for this), so a growing underestimate with $h$ is a property of the testbed, not
  only of a method.
- **Synthetic (`tree_source = "synthetic"`):** balanced trees of the three services of B.1, Poisson arrivals at
  `tree_utilization` of the slowest service.
- **Methods:** as in B.1 without DNN Sum; both versions of Pegasus.

Figures: `figures/tsc/b3_trees_real_<tree set>` (real) and `figures/tsc/b3_trees_synthetic` (tightness, training and
evaluation time).

In [2]:
# the options of the data source of B.3
tree_setup = configure(tree_source)
if "B.3" in tree_setup["skipped"]:
    print("B.3 skipped: not supported by the {} data".format(tree_source))
else:
    tree_levels = tree_setup["tree_levels"]
    tree_seeds = tree_setup["tree_seeds"]   # the seeds of every experiment
    # B.3: balanced fork-join trees, (shape, levels) -> (children, the service type or node of every position); the shape
    # "balanced" stays in the key, because the results are keyed by shape (sweep_statistics below)
    if tree_source == "synthetic":   # pegasus/trees.py
        trees = {("balanced", h): balanced_tree(h) for h in tree_levels}
        tree_experiments = None
    else:                            # the topology of every measured balanced tree (tree_structure in pegasus/real_data.py)
        tree_experiments = tree_setup["tree_experiments"]
        trees = {key: tree_structure(experiment)[:2] for key, experiment in tree_experiments.items()}
    # the synthetic trees reuse the configurations and the training data of the synthetic base case of B.1 (setup cell)
    tree_bases = bases if tree_source == "synthetic" else None

    def compute():
        def tree_data(shape, levels, s):
            children, path = trees[(shape, levels)]
            if tree_source == "real":   # the measured runs of the tree, every service trained on its benchmark
                return real_tree_data(tree_experiments[(shape, levels)], level=1 - alpha, node_level=1 - alpha / len(path),
                                      coverage_check=coverage_check, seed=[seed, s], **tree_setup["base_options"])
            data = synthetic_tree_data(children, path, level=1 - alpha, node_level=1 - alpha / len(path), coverage_check=coverage_check,
                                       seed=[seed, s], utilization=tree_setup["tree_utilization"], service_seed=s)
            # the configurations and training data of B.1, every configuration with one worker
            assert same_configurations(model_inputs(data.x_test, data.workers_column), model_inputs(tree_bases[s].x_test, tree_bases[s].workers_column)), \
                "the held-out configurations differ from those of B.1"
            assert all(np.array_equal(data.y_train[name], tree_bases[s].y_train[name]) for name in data.y_train), "the training data differs from that of B.1"
            assert_unseen(data)
            return data

        tree_truth, tree_methods, tree_seconds = {}, {}, {}   # shape -> (levels, seed) -> ...
        with warnings.catch_warnings():
            # for some services the fitted GP hyperparameters end at their bounds, and scikit-learn warns for every such model
            warnings.simplefilter("ignore", ConvergenceWarning)
            for s in tree_seeds:
                for shape, levels in trees:
                    data = tree_data(shape, levels, s)
                    models = fit_services(data, data.x_train, data.y_train, model_seed=seed + s, confidence=confidence, with_dnn=False)
                    tree_methods.setdefault(shape, {})[(levels, s)], tree_seconds.setdefault(shape, {})[(levels, s)] = compare_tree_methods(
                        data, models, alpha, confidence, n_calibration, model_seed=seed + s, leave_out=only_base_case)
                    tree_truth.setdefault(shape, {})[(levels, s)] = (data.measured, data.measured_low)
                print(f"seed {s}: {len(trees)} trees evaluated")
        return dict(tree_truth=tree_truth, tree_methods=tree_methods, tree_seconds=tree_seconds)

    globals().update(cached("b3_trees", tree_source, compute, inputs=dict(
        tree_levels=tree_levels, tree_seeds=tree_seeds, tree_options=dict(tree_utilization=tree_setup.get("tree_utilization"), tree_experiments=tree_experiments))))

seed 0: 4 trees evaluated
seed 1: 4 trees evaluated
seed 2: 4 trees evaluated
b3_trees: results computed and saved to statics/evaluation_cache/b3_trees_real+b1+b1u-max3_7f4659ad6f0ed130.joblib


In [ ]:
if "B.3" in tree_setup["skipped"]:
    print("B.3 skipped: not supported by the {} data".format(tree_source))
else:
    # the shapes (synthetic) or tree sets (real, VM_TREE_SETS in pegasus/real_data.py) in the order of `trees`, each with its levels h
    tree_shapes = {shape: [h for of, h in trees if of == shape] for shape, _ in trees}
    # the resolution range of a real tree set, from the experiment.yaml of its trees
    shape_text = {"balanced": "balanced", "large": "large trees (288 to 640 px)", "small": "small trees (160 to 256 px)"}
    tree_statistics, tree_times = {}, {}   # shape -> name -> coverage, tightness, holds, computed; shape -> name -> training, evaluation
    for shape, levels in tree_shapes.items():
        tree_statistics[shape] = sweep_statistics(tree_methods[shape], tree_truth[shape], levels, tree_seeds, confidence)
        tree_times[shape] = sweep_seconds(tree_seconds[shape], levels, tree_seeds)
        size_ticks = [f"{h} ({len(trees[(shape, h)][1])})" for h in levels]   # one row: the fixed layout of every figure
        if tree_source == "synthetic":
            title = (f"B.3 balanced fork-join trees, synthetic data: leaves s1, s2, aggregators s3, one worker each, "
                     f"{len(base.measured)} held-out configurations per seed,\n{workload['process']} arrivals at "
                     f"{tree_setup['tree_utilization']:.0%} utilization of the slowest service, {len(tree_seeds)} seeds")
        else:
            from pegasus.real_data import run_table
            # the utilization of the slowest service at which the campaign set the arrival rate of every run
            tree_utilization = sorted({float(u) for (of, _), experiment in tree_experiments.items() if of == shape
                                       for u in run_table(experiment)["utilization_asked"]})
            held_out = sorted({len(measured) for measured, _ in tree_truth[shape].values()})   # held-out runs of every tree
            runs_text = f"{held_out[0]}" if len(held_out) == 1 else f"{held_out[0]} to {held_out[-1]}"
            title = (f"B.3 balanced fork-join trees, real data, {shape_text[shape]}: leaves object and face detector, aggregators image "
                     f"transformer, one worker each, {runs_text} held-out runs per tree,\nPoisson arrivals at "
                     f"{' and '.join(f'{u:.0%}' for u in tree_utilization)} utilization of the slowest service, {len(tree_seeds)} seeds (training draws)")
        tightness_and_time_figure(levels, tree_statistics[shape], tree_times[shape], "levels h of the balanced tree (number of services)",
                                  title, p_label, tick_labels=size_ticks, log_x=False,
                                  name=f"b3_trees_{tree_source}" if shape == "balanced" else f"b3_trees_{tree_source}_{shape}",
                                  # the two time figures side by side in the width of one: too narrow for "h (N)" at every tick, and the
                                  # tightness figure already gives N for every h
                                  time_halves=("levels h of the balanced tree", [str(h) for h in levels]))

In [ ]:
if "B.3" in tree_setup["skipped"]:
    print("B.3 skipped: not supported by the {} data".format(tree_source))
else:
    for shape, levels in tree_shapes.items():
        tree_label = lambda h: f"h = {h} ({len(trees[(shape, h)][1])})"   # levels (number of services)
        print(f"{shape_text[shape].capitalize()} ({tree_source} data; tightness, median over the seeds; ● holds, ○ does not):")
        display(sweep_table(levels, tree_statistics[shape], sweep_competitors + ["Oracle"], tree_label, tree_truth[shape], tree_seeds))
        print("Training and evaluation time (s, median over the seeds):")
        display(time_table(levels, tree_times[shape], sweep_competitors, tree_label))